### **Enviornment**

In [2]:
# Setting up the local imports and env variables
import os 
from dotenv import load_dotenv
load_dotenv()

os.environ['LANGCHAIN_TRACING_V2'] = os.getenv("LANGCHAIN_TRACING_V2")
os.environ['LANGCHAIN_ENDPOINT'] = os.getenv("LANGCHAIN_ENDPOINT")
os.environ['LANGCHAIN_API_KEY'] = os.getenv("LANGCHAIN_API_KEY")
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")

### **Overview**

In [13]:
# Import statements 

import bs4
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI , OpenAIEmbeddings
from langsmith import Client
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import OllamaEmbeddings , ChatOllama

# Indexing 
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)
docs = loader.load()

# Split 
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 1000 , chunk_overlap = 200)
splits = text_splitter.split_documents(docs)

# Embed
embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)
vectorstore = Chroma.from_documents(documents = splits , embedding=embeddings)
retriver = vectorstore.as_retriever()

# Retrival and Generation
llm = ChatOllama(model="llama3.2:3b", temperature=0.2)

# prompt 
prompt = ChatPromptTemplate.from_template("""
You are an assistant for question-answering tasks.

Use the following retrieved context to answer the question.
If the answer isn't in the context, say you don't know.

Context:
{context}

Question:
{question}
""")

def format_doc(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# chain
rag_chain = (
    {"context" : retriver | format_doc , "question" : RunnablePassthrough()}
    | prompt | llm | StrOutputParser()
)

rag_chain.invoke("What is Task Decomposition?")


'Task decomposition is the process of breaking down a complicated task into smaller, simpler steps. It involves identifying the individual tasks that need to be completed to achieve the overall goal and planning ahead to ensure each step is taken correctly.'

### **Indexing**

In [8]:
# Documents
question = "What kinds of pets do i like?"
document = "My Favourite pet is a cat."

In [9]:
import tiktoken

def num_tokens_from_string(string : str , encoding_name : str) -> int:
    """Returns the number of tokens in a text string"""
    encoding = tiktoken.get_encoding(encoding_name)
    num_tokens = len(encoding.encode(string))
    return num_tokens

num_tokens_from_string(question , "cl100k_base")

8

In [11]:
from langchain_ollama import OllamaEmbeddings
embed = OllamaEmbeddings(model="nomic-embed-text")
query_result = embed.embed_query(question)
document_rsult = embed.embed_query(document)
len(query_result) , len(document_rsult)

(768, 768)

In [12]:
import numpy as np 

def cosine_similatiry(vec1 , vec2):
    dot_product = np.dot(vec1 , vec2)
    norm_vec1 = np.linalg.norm(vec1)
    norm_vec2 = np.linalg.norm(vec2)
    return dot_product / (norm_vec1 * norm_vec2)

similarity = cosine_similatiry(query_result , document_rsult)
print("Cosine similarity :- " , similarity)

Cosine similarity :-  0.7578530323544367
